# Baseline (Fase 2): YOLOv8s @ 640

Entrenamiento baseline con configuración por defecto. Establece la referencia honesta para comparar los experimentos de la fase 3. Hiperparámetros en `configs/hyp_baseline.yaml`.

> **Notebook delgado**: solo orquesta. La lógica vive en `src/` del repositorio.

In [ ]:
# Setup en Kaggle: clona el repo del proyecto e instala dependencias.
# El código pesado vive en `src/`; este notebook solo orquesta.
import os, subprocess, sys

REPO_URL = "https://github.com/<usuario>/<repo>.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Carga el archivo de hiperparámetros del experimento (hyp_baseline.yaml).
# Sobrescribe `path` de data.yaml para apuntar al Kaggle Dataset montado.
import yaml, shutil
from pathlib import Path

HYP_FILE = Path("configs/hyp_baseline.yaml")
DATA_YAML = Path("configs/data.yaml")
# TODO: reemplazar por la ruta real del Kaggle Dataset montado, por ejemplo:
#   /kaggle/input/epp-yolo-processed
KAGGLE_DATA_ROOT = "/kaggle/input/epp-yolo-processed"

with HYP_FILE.open() as f:
    hyp = yaml.safe_load(f)

data_cfg = yaml.safe_load(DATA_YAML.read_text())
data_cfg["path"] = KAGGLE_DATA_ROOT
patched_data = Path("/kaggle/working/data_patched.yaml")
patched_data.write_text(yaml.safe_dump(data_cfg, sort_keys=False))

print("Hiperparámetros:", hyp)
print("data.yaml parcheado:", patched_data)

In [ ]:
# Entrenamiento. Usa la API de Ultralytics directamente; toda la
# justificación de hiperparámetros vive en el archivo YAML cargado.
from ultralytics import YOLO

model = YOLO(hyp["model"])
results = model.train(
    data=str(patched_data),
    name="baseline",
    project="/kaggle/working/runs",
    **{k: v for k, v in hyp.items() if k != 'model'},
)

# Los artefactos quedan en /kaggle/working/runs/<name>/. Descargar el
# best.pt al final del run y publicarlo como release del repo Git.

In [ ]:
# Listar artefactos producidos para descarga (no commitear los .pt).
from pathlib import Path
run_dir = Path("/kaggle/working/runs/baseline")
for p in sorted(run_dir.rglob('*')):
    if p.is_file():
        print(p)